# Model 3: Oil-Augmented Volatility

**Core thesis test:** Does incorporating oil price volatility improve the model for oil exporters?

**Specification:**
```
Oil exporters: σ²_oil = σ²_garch + θ × σ²_brent
Controls:      σ²_oil = σ²_garch  (no oil component)
```

**Diff-in-diff logic:**
```
DiD = (R²_oil - R²_garch)_oil_exporters - (R²_oil - R²_garch)_controls
```

Since controls have σ_oil = σ_garch, their improvement should be ~0.
If DiD > 0 → Adding oil vol helps oil exporters (supports thesis)

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm, linregress
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from arch import arch_model

In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================
PANEL_PATH = "data/processed/merton_panel.csv"
BRENT_PATH = "data/processed/Commodities/brent_weekly.csv"

T = 1.0
TARGET_RATIO = 1.5
THETA = 1.0  # Oil volatility transmission parameter

---
## 1. Load Panel

In [4]:
panel = pd.read_csv(PANEL_PATH, parse_dates=['date'])
panel = panel.sort_values(['country', 'date']).reset_index(drop=True)

print(f"Panel: {len(panel)} obs, {panel['country'].nunique()} countries")
print(f"\nGroups:")
print(panel.groupby('group')['country'].nunique())

Panel: 10961 obs, 20 countries

Groups:
group
control         8
oil_exporter    6
other           6
Name: country, dtype: int64


In [5]:
print("Oil Exporters:")
print(list(panel[panel['group']=='oil_exporter']['country'].unique()))
print("\nControls:")
print(list(panel[panel['group']=='control']['country'].unique()))

Oil Exporters:
['Brazil', 'Colombia', 'Egypt', 'Malaysia', 'Mexico', 'Saudi Arabia']

Controls:
['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa', 'South Korea', 'Thailand', 'Turkey']


---
## 2. Load Brent

In [6]:
try:
    brent = pd.read_csv(BRENT_PATH, parse_dates=['date'])
    print(f"Loaded Brent: {len(brent)} obs")
except FileNotFoundError:
    print("Brent file not found. Creating synthetic data.")
    print("WARNING: Replace with real Brent prices!")
    
    dates = panel['date'].drop_duplicates().sort_values()
    np.random.seed(42)
    n = len(dates)
    
    returns = np.random.normal(0, 0.04, n)
    jump_idx = np.random.choice(n, size=int(n*0.03), replace=False)
    returns[jump_idx] += np.random.choice([-0.15, -0.10, 0.08, 0.10], len(jump_idx))
    
    prices = 70 * np.exp(np.cumsum(returns))
    brent = pd.DataFrame({'date': dates, 'brent_price': prices})

Brent file not found. Creating synthetic data.


In [7]:
brent = brent.sort_values('date')
brent['brent_ret'] = np.log(brent['brent_price'] / brent['brent_price'].shift(1))
print(f"Brent returns: mean={brent['brent_ret'].mean():.4f}, std={brent['brent_ret'].std():.4f}")

Brent returns: mean=-0.0012, std=0.0426


---
## 3. Fit GARCH

In [8]:
def fit_garch(returns, annualize=True):
    returns = returns.dropna()
    if len(returns) < 100:
        return None, None
    
    returns_pct = returns * 100
    try:
        model = arch_model(returns_pct, vol='Garch', p=1, q=1, mean='Zero', rescale=False)
        result = model.fit(disp='off', show_warning=False)
        
        sigma = result.conditional_volatility / 100
        if annualize:
            sigma = sigma * np.sqrt(52)
        
        params = {
            'alpha': result.params.get('alpha[1]', np.nan),
            'beta': result.params.get('beta[1]', np.nan),
        }
        return sigma, params
    except:
        return None, None

In [9]:
# Brent GARCH
brent_sigma, brent_params = fit_garch(brent['brent_ret'])

if brent_sigma is not None:
    brent['brent_vol'] = brent_sigma.values
    print(f"Brent GARCH: α={brent_params['alpha']:.3f}, β={brent_params['beta']:.3f}")
    print(f"Brent vol: mean={brent['brent_vol'].mean():.3f}")
else:
    brent['brent_vol'] = brent['brent_ret'].rolling(52, min_periods=12).std() * np.sqrt(52)
    print("Using rolling vol for Brent")

ValueError: Length of values (573) does not match length of index (574)

In [ ]:
# Country GARCH
panel['sigma_garch'] = np.nan

for country in panel['country'].unique():
    mask = panel['country'] == country
    returns = panel.loc[mask, 'msci_ret_weekly'].copy()
    
    sigma, params = fit_garch(returns)
    
    if sigma is not None:
        panel.loc[mask, 'sigma_garch'] = sigma.values
        print(f"{country}: α={params['alpha']:.3f}, β={params['beta']:.3f}")
    else:
        print(f"{country}: GARCH failed")

---
## 4. Merge Brent and Compute Oil-Augmented Vol

In [ ]:
panel = panel.merge(brent[['date', 'brent_vol']], on='date', how='left')
panel['brent_vol'] = panel['brent_vol'].ffill()

In [ ]:
# =============================================================================
# OIL-AUGMENTED VOLATILITY
# Oil exporters: σ²_oil = σ²_garch + θ × σ²_brent
# Controls:      σ²_oil = σ²_garch
# =============================================================================

# Create oil dummy
panel['is_oil'] = (panel['group'] == 'oil_exporter').astype(int)

# Oil-augmented vol: only add brent vol for oil exporters
panel['sigma_oil'] = np.sqrt(
    panel['sigma_garch']**2 + 
    panel['is_oil'] * THETA * panel['brent_vol']**2
)

print("Volatility by group:")
print(panel.groupby('group')[['sigma_garch', 'sigma_oil', 'brent_vol']].mean().round(4))

In [ ]:
# Verify: controls should have sigma_oil == sigma_garch
ctrl_check = panel[panel['group'] == 'control']
diff = (ctrl_check['sigma_oil'] - ctrl_check['sigma_garch']).abs().mean()
print(f"Controls: mean |sigma_oil - sigma_garch| = {diff:.6f} (should be ~0)")

---
## 5. Compute d₂ and PD

In [ ]:
# Scale assets
scale_factors = panel.groupby('country').apply(
    lambda g: TARGET_RATIO * g['default_barrier'].mean() / g['msci_index'].mean()
).reset_index()
scale_factors.columns = ['country', 'scale_factor']

panel = panel.merge(scale_factors, on='country', how='left')
panel['A'] = panel['msci_index'] * panel['scale_factor']

In [ ]:
def compute_d2(A, B, sigma, r, T=1.0):
    with np.errstate(divide='ignore', invalid='ignore'):
        d2 = (np.log(A / B) + (r - sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    return d2

def compute_pd(d2):
    return norm.cdf(-d2)

In [ ]:
# Model 1: Baseline (52w rolling)
panel['d2_baseline'] = compute_d2(panel['A'], panel['default_barrier'], 
                                   panel['msci_vol_52w'], panel['rf'], T)
panel['pd_baseline'] = compute_pd(panel['d2_baseline'])

# Model 2: GARCH
panel['d2_garch'] = compute_d2(panel['A'], panel['default_barrier'], 
                                panel['sigma_garch'], panel['rf'], T)
panel['pd_garch'] = compute_pd(panel['d2_garch'])

# Model 3: Oil-augmented (only affects oil exporters)
panel['d2_oil'] = compute_d2(panel['A'], panel['default_barrier'], 
                              panel['sigma_oil'], panel['rf'], T)
panel['pd_oil'] = compute_pd(panel['d2_oil'])

---
## 6. Compute Changes for Regression

In [ ]:
panel = panel.replace([np.inf, -np.inf], np.nan)
panel = panel.dropna(subset=['d2_baseline', 'd2_garch', 'd2_oil',
                              'pd_baseline', 'pd_garch', 'pd_oil'])

panel = panel.sort_values(['country', 'date'])

# Changes
panel['ln_cds'] = np.log(panel['cds_spread'])
panel['d_ln_cds'] = panel.groupby('country')['ln_cds'].diff()

panel['d_pd_baseline'] = panel.groupby('country')['pd_baseline'].diff()
panel['d_pd_garch'] = panel.groupby('country')['pd_garch'].diff()
panel['d_pd_oil'] = panel.groupby('country')['pd_oil'].diff()

panel['ym'] = panel['date'].dt.to_period('M').astype(str)

reg_data = panel.dropna(subset=['d_ln_cds', 'd_pd_baseline', 'd_pd_garch', 'd_pd_oil'])
reg_data = reg_data.replace([np.inf, -np.inf], np.nan).dropna(
    subset=['d_ln_cds', 'd_pd_baseline', 'd_pd_garch', 'd_pd_oil']
)

print(f"Regression sample: {len(reg_data)} obs, {reg_data['country'].nunique()} countries")

---
## 7. Full Sample Results

In [ ]:
m1 = smf.ols('d_ln_cds ~ d_pd_baseline + C(country) + C(ym)', data=reg_data).fit()
m2 = smf.ols('d_ln_cds ~ d_pd_garch + C(country) + C(ym)', data=reg_data).fit()
m3 = smf.ols('d_ln_cds ~ d_pd_oil + C(country) + C(ym)', data=reg_data).fit()

print("="*80)
print("FULL SAMPLE: THREE-MODEL COMPARISON")
print("="*80)
print(f"{'Metric':<15} {'Baseline':>20} {'GARCH':>20} {'Oil-Augmented':>20}")
print("-"*80)
print(f"{'β (ΔPD)':<15} {m1.params['d_pd_baseline']:>20.4f} {m2.params['d_pd_garch']:>20.4f} {m3.params['d_pd_oil']:>20.4f}")
print(f"{'t-stat':<15} {m1.tvalues['d_pd_baseline']:>20.2f} {m2.tvalues['d_pd_garch']:>20.2f} {m3.tvalues['d_pd_oil']:>20.2f}")
print(f"{'R²':<15} {m1.rsquared:>20.4f} {m2.rsquared:>20.4f} {m3.rsquared:>20.4f}")

---
## 8. KEY TEST: By Group

In [ ]:
oil_sample = reg_data[reg_data['group'] == 'oil_exporter']
ctrl_sample = reg_data[reg_data['group'] == 'control']

print(f"Oil exporters: {oil_sample['country'].nunique()} countries, {len(oil_sample)} obs")
print(f"Controls: {ctrl_sample['country'].nunique()} countries, {len(ctrl_sample)} obs")

In [ ]:
group_results = []

for group_name, sample in [('Oil Exporters', oil_sample), ('Controls', ctrl_sample)]:
    m1 = smf.ols('d_ln_cds ~ d_pd_baseline + C(country) + C(ym)', data=sample).fit()
    m2 = smf.ols('d_ln_cds ~ d_pd_garch + C(country) + C(ym)', data=sample).fit()
    m3 = smf.ols('d_ln_cds ~ d_pd_oil + C(country) + C(ym)', data=sample).fit()
    
    group_results.append({
        'group': group_name,
        'n_countries': sample['country'].nunique(),
        'r2_baseline': m1.rsquared,
        'r2_garch': m2.rsquared,
        'r2_oil': m3.rsquared,
        'delta_oil_vs_garch': m3.rsquared - m2.rsquared
    })

group_df = pd.DataFrame(group_results)
print(group_df.to_string(index=False))

In [ ]:
# DIFF-IN-DIFF
oil_row = group_df[group_df['group'] == 'Oil Exporters'].iloc[0]
ctrl_row = group_df[group_df['group'] == 'Controls'].iloc[0]

did = oil_row['delta_oil_vs_garch'] - ctrl_row['delta_oil_vs_garch']

print("\n" + "="*60)
print("★ DIFFERENCE-IN-DIFFERENCES (KEY THESIS RESULT) ★")
print("="*60)
print(f"\nOil exporters (R²_oil - R²_garch): {oil_row['delta_oil_vs_garch']:+.4f}")
print(f"Controls (R²_oil - R²_garch):      {ctrl_row['delta_oil_vs_garch']:+.4f}")
print(f"\nDIFF-IN-DIFF: {did:+.4f}")
print("\nNote: Controls should be ~0 since σ_oil = σ_garch for them")

---
## 9. Country-Level

In [ ]:
country_results = []

for country in reg_data['country'].unique():
    df = reg_data[reg_data['country'] == country]
    if len(df) < 50:
        continue
    
    _, _, r1, _, _ = linregress(df['d_pd_baseline'], df['d_ln_cds'])
    _, _, r2, _, _ = linregress(df['d_pd_garch'], df['d_ln_cds'])
    _, _, r3, _, _ = linregress(df['d_pd_oil'], df['d_ln_cds'])
    
    country_results.append({
        'country': country,
        'group': df['group'].iloc[0],
        'r2_baseline': r1**2,
        'r2_garch': r2**2,
        'r2_oil': r3**2,
        'delta_oil_vs_garch': r3**2 - r2**2
    })

country_df = pd.DataFrame(country_results).sort_values('delta_oil_vs_garch', ascending=False)
print(country_df.to_string(index=False))

In [ ]:
print("\nMean improvement by group:")
print(country_df.groupby('group')['delta_oil_vs_garch'].mean().round(4))

---
## 10. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. R² by group
ax = axes[0, 0]
x = np.arange(2)
w = 0.25
ax.bar(x - w, group_df['r2_baseline'], w, label='Baseline')
ax.bar(x, group_df['r2_garch'], w, label='GARCH')
ax.bar(x + w, group_df['r2_oil'], w, label='Oil-Augmented', color='green')
ax.set_xticks(x)
ax.set_xticklabels(group_df['group'])
ax.set_ylabel('R²')
ax.set_title('R² by Group')
ax.legend()

# 2. Diff-in-diff
ax = axes[0, 1]
bars = ax.bar(['Oil Exporters', 'Controls'], 
              [oil_row['delta_oil_vs_garch'], ctrl_row['delta_oil_vs_garch']],
              color=['orange', 'blue'], alpha=0.8)
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_ylabel('R² Improvement (Oil vs GARCH)')
ax.set_title(f'Diff-in-Diff = {did:+.4f}')
for bar, val in zip(bars, [oil_row['delta_oil_vs_garch'], ctrl_row['delta_oil_vs_garch']]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:+.4f}', ha='center', fontweight='bold')

# 3. Country bar chart
ax = axes[1, 0]
country_sorted = country_df.sort_values('delta_oil_vs_garch', ascending=True)
colors = ['orange' if g == 'oil_exporter' else 'blue' for g in country_sorted['group']]
ax.barh(country_sorted['country'], country_sorted['delta_oil_vs_garch'], color=colors, alpha=0.7)
ax.axvline(0, color='k', linestyle='-', linewidth=0.5)
ax.set_xlabel('R² Improvement (Oil vs GARCH)')
ax.set_title('By Country (Orange=Oil, Blue=Control)')

# 4. Time series example
ax = axes[1, 1]
# Pick an oil exporter
sample_country = oil_sample['country'].unique()[0]
sample = panel[panel['country'] == sample_country]
ax.plot(sample['date'], sample['sigma_garch'], label='GARCH', alpha=0.8)
ax.plot(sample['date'], sample['sigma_oil'], label='Oil-Augmented', alpha=0.8, linestyle='--')
ax.plot(sample['date'], sample['brent_vol'], label='Brent', alpha=0.5, color='brown')
ax.set_ylabel('Volatility')
ax.set_title(f'{sample_country}: Volatility Comparison')
ax.legend()

plt.tight_layout()
plt.show()

---
## 11. Summary

In [ ]:
print("\n" + "="*70)
print("MODEL 3: OIL-AUGMENTED VOLATILITY — SUMMARY")
print("="*70)

print("\n1. SPECIFICATION")
print("   Oil exporters: σ²_oil = σ²_garch + θ × σ²_brent")
print("   Controls:      σ²_oil = σ²_garch")
print(f"   θ = {THETA}")

print("\n2. KEY RESULT")
print(f"   Oil exporters improvement: {oil_row['delta_oil_vs_garch']:+.4f}")
print(f"   Controls improvement:      {ctrl_row['delta_oil_vs_garch']:+.4f}")
print(f"   DIFF-IN-DIFF:              {did:+.4f}")

print("\n3. INTERPRETATION")
if did > 0.005:
    print("   ✓ SUPPORTS THESIS")
    print("   ✓ Adding oil volatility improves model for oil exporters")
elif did > 0:
    print("   ~ Weak support (small positive effect)")
else:
    print("   ✗ Does not support thesis")

---
## 12. Save

In [ ]:
group_df.to_csv('data/processed/model3_group_results.csv', index=False)
country_df.to_csv('data/processed/model3_country_results.csv', index=False)

metrics = {
    'model': 'Oil-Augmented',
    'theta': THETA,
    'r2_oil_exporters_garch': oil_row['r2_garch'],
    'r2_oil_exporters_oil': oil_row['r2_oil'],
    'r2_controls_garch': ctrl_row['r2_garch'],
    'r2_controls_oil': ctrl_row['r2_oil'],
    'diff_in_diff': did
}
pd.DataFrame([metrics]).to_csv('data/processed/model3_metrics.csv', index=False)

print("Saved results.")